# Gaming Toxicity Detection

**Authors:** Beibarys Nyussupov, Ruby Ngo, Paola Calle, Jett Forward

In [1]:
# libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path 
import re 
import sys
sys.path.insert(0, str(Path("../..").resolve()))

# custom functions - tokenizer, step evaluator
from src.tokenizer import tokenize
from src.prep_evaluation_multi import eval_step

# to split the data into train and test
from sklearn.model_selection import train_test_split

# reproducibility
seed = 7524
np.random.seed(seed)

# project root (notebooks/gaming/ - notebooks/ - project root)
PROJECT_ROOT = Path().resolve().parent.parent

In [2]:
# data directories
DATA_DIR_WOT  = PROJECT_ROOT / "data/raw_data/wot/"
DATA_DIR_DOTA = PROJECT_ROOT / "data/raw_data/dota/"

## World of Tanks

**World of Tanks**
| Class | Terminology |
|---|---|
| 0 | Non-Toxic |
| 1 | Insults and Flaming |
| 2 | Other Offensive Texts |
| 3 | Hate and Harassment |
| 4 | Threats |
| 5 | Extremism |

#### Inspect each split

In [3]:
# import data 
train = pd.read_csv(DATA_DIR_WOT / "train.csv")
val = pd.read_csv(DATA_DIR_WOT / "val.csv")

# test dataset 
test_text = pd.read_csv(DATA_DIR_WOT /  "test_index_text.csv")
test_label = pd.read_csv(DATA_DIR_WOT / "test_index_label.csv")

# check the data 
for name, df in [("train", train), ("val", val), ("test_text", test_text), ("test_label", test_label)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"First few rows of the dataset:\n{df.head(3)}")
    print()

--- train ---
Shape: (42959, 3)
Columns: ['index', 'message', 'label']
First few rows of the dataset:
   index                        message  label
0  30702                        no rush    0.0
1  18607  whatever ... watch the replay    0.0
2  32901                        useless    1.0

--- val ---
Shape: (5367, 3)
Columns: ['index', 'message', 'label']
First few rows of the dataset:
   index       message  label
0  31697  e50 is there    0.0
1  52311   this scouts    0.0
2   2775       i guess    0.0

--- test_text ---
Shape: (5375, 2)
Columns: ['index', 'message']
First few rows of the dataset:
   index         message
0  17775  aim you monkey
1  28228   Pz kill artys
2  19888             ggs

--- test_label ---
Shape: (5375, 2)
Columns: ['index', 'label']
First few rows of the dataset:
   index  label
0  17775    2.0
1  28228    0.0
2  19888    0.0



The test set is stored in two separate files - text and labels indexed by the same `index` column. Train and val are already complete. We need to join the test files on `index` before merging everything.

#### Reconstruct test split and merge all

In [4]:
# merge test text and labels
test = test_text.merge(test_label, on="index", how="inner")

# concatenate all data
wot = pd.concat([train, val, test], ignore_index=True)

print(f"Test join shape: {test.shape}\n")
print(f"Merged shape: {wot.shape}\n")
print(f"First few rows of the data:\n{wot.head(3)}\n")
print(f"Information about the data: {wot.info()}")

Test join shape: (5375, 3)

Merged shape: (53701, 3)

First few rows of the data:
   index                        message  label
0  30702                        no rush    0.0
1  18607  whatever ... watch the replay    0.0
2  32901                        useless    1.0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53701 entries, 0 to 53700
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   index    53701 non-null  int64  
 1   message  53701 non-null  object 
 2   label    53701 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.2+ MB
Information about the data: None


The test join uses `inner` - any index present in only one file would be silently dropped. If test shape after join is smaller than either file's length, there is an alignment issue in the original data. 

In [5]:
# X and Y
X = wot["message"]
y = wot["label"]

# split the data into our train and test 
x_train, x_validation, y_train, y_validation = train_test_split(X, y, test_size=0.2, random_state=seed, 
                                             shuffle = True, stratify=y)

# distributions for each split 
print("Distribution of labels in the training set normalized and not-normalized:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))
print("\nDistribution of labels in the validation set normalized and not-normalized:")
print(y_validation.value_counts())
print(y_validation.value_counts(normalize=True))

# save both datasets as one parquet file each, with message and label columns,
# for easier processing in the next steps
x_train = pd.DataFrame({"message": x_train, "label": y_train})
x_validation = pd.DataFrame({"message": x_validation, "label": y_validation})
x_train.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_train.parquet", index=False)
x_validation.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_validation.parquet", index=False)

Distribution of labels in the training set normalized and not-normalized:
label
0.0    34797
1.0     5926
2.0     1874
3.0      279
4.0       60
5.0       24
Name: count, dtype: int64
label
0.0    0.809986
1.0    0.137942
2.0    0.043622
3.0    0.006494
4.0    0.001397
5.0    0.000559
Name: proportion, dtype: float64

Distribution of labels in the validation set normalized and not-normalized:
label
0.0    8700
1.0    1481
2.0     469
3.0      70
4.0      15
5.0       6
Name: count, dtype: int64
label
0.0    0.809980
1.0    0.137883
2.0    0.043664
3.0    0.006517
4.0    0.001397
5.0    0.000559
Name: proportion, dtype: float64


In [6]:
# baseline - before any cleaning
impact = None
impact = eval_step("baseline", {"WoT": PROJECT_ROOT / "data/processed_data/wot/x_train.parquet"}, impact)
impact

    step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
baseline     WoT 42960    0.4743       0.8646        0.5476           0.4626             0.0           0.0              0.0


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,WoT,42960,0.4743,0.8646,0.5476,0.4626,0.0,0.0,0.0


#### Non-english language 

Since we are going to model classifier on english data, it is important for us to detect non-english cases as many as we can. 

In [7]:
# detecting Non-lating-script messages 

# covers all major non-Latin characters
NON_LATIN_SCRIPT = re.compile(
    r"[\u0400-\u04FF"   # Cyrillic
    r"\u4E00-\u9FFF"    # CJK unified ideographs
    r"\u3400-\u4DBF"    # CJK extension A
    r"\uF900-\uFAFF"    # CJK compatibility ideographs
    r"\u0600-\u06FF"    # Arabic
    r"\u0590-\u05FF"    # Hebrew
    r"\u3040-\u30FF"    # Japanese (Hiragana + Katakana)
    r"\uAC00-\uD7AF"    # Korean (Hangul syllables)
    r"\u1100-\u11FF"    # Korean (Hangul Jamo)
    r"\u0E00-\u0E7F"    # Thai
    r"\u0900-\u097F"    # Devanagari (Hindi)
    r"\u0980-\u09FF"    # Bengali
    r"\u0370-\u03FF"    # Greek
    r"\u10A0-\u10FF"    # Georgian
    r"\u0530-\u058F"    # Armenian
    r"\u1000-\u109F"    # Myanmar
    r"\u1780-\u17FF]"   # Khmer
)

# drop any message containing even 1 non-Latin script char
print(f"Non-english documents:\n{x_train[x_train['message'].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)}\n")

# changes 
before = len(x_train)
x_train_en = x_train[~x_train["message"].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)
print(f"Dropped {before - len(x_train_en)} ({(before - len(x_train_en))/before:.1%})")
print("Label distribution before:")
print(x_train["label"].value_counts(normalize=True).sort_index(), "\n")

print("\nLabel distribution after:")
print(x_train_en["label"].value_counts(normalize=True).sort_index(), "\n")
print(f"Rows after the drop: {x_train_en.shape[0]}")
# save new step again
x_train_en.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_train.parquet", index=False)

Non-english documents:
                                    message  label
0     по крайслеру ніхто постріляти не може    0.0
1                   Засвет в квадрате J4!!!    0.0
2                              пидоры тупые    1.0
3                     ору в голосину просто    0.0
4                            сыны пидорасов    3.0
...                                     ...    ...
3058                Засвет в квадрате K8!!!    0.0
3059                                   сука    1.0
3060                        они все наверху    0.0
3061                          низкий поклон    0.0
3062                        ВСЕ НАДЕРЕЖАБЛЬ    1.0

[3063 rows x 2 columns]

Dropped 3063 (7.1%)
Label distribution before:
label
0.0    0.809986
1.0    0.137942
2.0    0.043622
3.0    0.006494
4.0    0.001397
5.0    0.000559
Name: proportion, dtype: float64 


Label distribution after:
label
0.0    0.812567
1.0    0.136176
2.0    0.043863
3.0    0.005464
4.0    0.001353
5.0    0.000576
Name: proportion, dtype: f

In [8]:
# check non-latin language drop
impact = eval_step("non_latin_drop", {"WoT": PROJECT_ROOT / "data/processed_data/wot/x_train.parquet"}, impact)
impact

          step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
non_latin_drop     WoT 39897     0.493       0.8699        0.5725           0.4929          0.0187        0.0249           0.0303


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,WoT,42960,0.4743,0.8646,0.5476,0.4626,0.0000,0.0000,0.0000
1,non_latin_drop,WoT,39897,0.4930,0.8699,0.5725,0.4929,0.0187,0.0249,0.0303


In [9]:
# transform that on the validation set as well, to be consistent with the training data.
# drop any message containing even 1 non-Latin script char
print(f"Non-english documents:\n{x_validation[x_validation['message'].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)}\n")

# changes 
before = len(x_validation)
x_validation_en = x_validation[~x_validation["message"].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)
print(f"Dropped {before - len(x_validation_en)} ({(before - len(x_validation_en))/before:.1%})")
print("Label distribution before:")
print(x_validation["label"].value_counts(normalize=True).sort_index(), "\n")

print("\nLabel distribution after:")
print(x_validation_en["label"].value_counts(normalize=True).sort_index(), "\n")
print(f"Rows after the drop: {x_validation_en.shape[0]}")
# save new step again
x_validation_en.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_validation.parquet", index=False)

Non-english documents:
                                       message  label
0                    кто берет в рот тот знает    0.0
1                          він просто пездоліс    1.0
2                                        т95((    0.0
3                                        козлт    0.0
4                                        блять    1.0
..                                         ...    ...
782                                  ТИГР СУКА    1.0
783                    Засвет в квадрате E8!!!    0.0
784                            вбийте дпібного    0.0
785                                 Так точно!    0.0
786  ну писать то он может, значит с монитором    0.0

[787 rows x 2 columns]

Dropped 787 (7.3%)
Label distribution before:
label
0.0    0.809980
1.0    0.137883
2.0    0.043664
3.0    0.006517
4.0    0.001397
5.0    0.000559
Name: proportion, dtype: float64 


Label distribution after:
label
0.0    0.813643
1.0    0.135925
2.0    0.043400
3.0    0.005023
4.0    0.001507
5.0    0

In [11]:
from lingua import Language, LanguageDetectorBuilder

# detecting Latin-script but likely non-English messages

# use all Latin-script languages supported by Lingua
# English must stay inside the detector, otherwise English text gets forced into another language
LATIN_LANGUAGES = list(Language.all_with_latin_script())

# detector for Latin-script language detection
# minimum_relative_distance: if top languages are too close, Lingua returns None instead of guessing
detector = (
    LanguageDetectorBuilder
    .from_languages(*LATIN_LANGUAGES)
    .with_minimum_relative_distance(0.25)
    .build()
)

# safety tokens: if these appear, keep the message because it may be gaming/chat English or mixed slang
SAFETY_TOKENS = {
    # gaming / chat
    "gg", "ez", "wp", "gl", "hf", "glhf", "afk", "brb", "lol", "lmao", "lmfao",
    "omg", "wtf", "stfu", "noob", "n00b", "bot", "team", "report", "ban",
    "kick", "mute", "trash", "toxic", "troll", "feed", "feeder", "int", "throw",
    "rank", "elo", "kill", "kys", "die", "fps", "lag", "ping", "server",
    "wot", "dota", "fortnite", "fn",

    # English / toxicity cues
    "you", "your", "you're", "u", "ur", "are", "is", "this", "that", "the",
    "and", "not", "dont", "don't", "fuck", "fucking", "fuk", "fck", "shit",
    "bitch", "ass", "asshole", "idiot", "stupid", "dumb", "moron", "loser",
    "suck", "sucks"
}

TOKEN_RE = re.compile(r"[a-z0-9']+", flags=re.IGNORECASE)


def get_tokens(text):
    return set(TOKEN_RE.findall(str(text).lower()))


def has_safety_token(text):
    return bool(get_tokens(text) & SAFETY_TOKENS)


# function to classify what to do with each Latin-script message
def latin_language_filter_status(text):
    text_str = str(text).strip()
    tokens = get_tokens(text_str)

    # too short - keep
    # language detection is unreliable on short gaming/chat messages
    if len(text_str) < 50 or len(tokens) < 6:
        return "keep_short"

    # possible gaming/chat/English false positive - keep
    if has_safety_token(text_str):
        return "keep_possible_gaming_or_mixed"

    lang = detector.detect_language_of(text_str)

    # ambiguous - keep
    if lang is None:
        return "keep_ambiguous"

    # English - keep
    if lang == Language.ENGLISH:
        return "keep_english"

    # confident Latin-script non-English - drop
    return f"drop_latin_nonenglish_{lang.name.lower()}"


# apply detection
x_train_en_lang = x_train_en.copy()
x_train_en_lang["lang_filter_status"] = x_train_en_lang["message"].apply(latin_language_filter_status)

# inspect groups
print("Language filter status counts:")
print(x_train_en_lang["lang_filter_status"].value_counts().head(30))


# check ambiguous cases
# these are messages where Lingua did not confidently choose a language
ambiguous_mask = x_train_en_lang["lang_filter_status"].eq("keep_ambiguous")

print(f"\nAmbiguous Latin-script language documents kept: {ambiguous_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nAmbiguous documents kept for now:\n",
        x_train_en_lang.loc[
            ambiguous_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among ambiguous rows:")
print(
    x_train_en_lang.loc[ambiguous_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# check possible gaming / mixed-language false positives
# these are messages that Lingua might classify as non-English, but contain gaming/chat/English cues
mixed_mask = x_train_en_lang["lang_filter_status"].eq("keep_possible_gaming_or_mixed")

print(f"\nPossible gaming or mixed-language documents kept: {mixed_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nPossible gaming/mixed documents kept for now:\n",
        x_train_en_lang.loc[
            mixed_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among possible gaming/mixed rows:")
print(
    x_train_en_lang.loc[mixed_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# drop only confident Latin-script non-English rows
drop_mask = x_train_en_lang["lang_filter_status"].str.startswith("drop_latin_nonenglish")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nConfident Latin-script non-English documents to drop:\n",
        x_train_en_lang.loc[
            drop_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print(f"\nHow many confident Latin-script non-English documents: {drop_mask.sum()}")


# label distribution sanity check
dropped_latin_nonenglish = x_train_en_lang.loc[drop_mask]

print("\nLabel distribution before Latin non-English drop:")
print(x_train_en["label"].value_counts(normalize=True).sort_index())

print("\nLabel distribution after Latin non-English drop:")
print(
    x_train_en_lang.loc[~drop_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nLabel distribution among dropped rows:")
print(
    dropped_latin_nonenglish["label"]
    .value_counts(normalize=True)
    .sort_index()
)


# create dataset after Latin non-English drop
# keep ambiguous and mixed/gaming rows
x_train_lang_clean = (
    x_train_en_lang
    .loc[~drop_mask]
    .drop(columns=["lang_filter_status"])
    .reset_index(drop=True)
)

print(f"\nRows before Latin non-English drop: {len(x_train_en_lang)}")
print(f"Dropped {drop_mask.sum()} ({drop_mask.sum() / len(x_train_en_lang):.1%})")
print(f"Rows after Latin non-English drop: {len(x_train_lang_clean)}")

Language filter status counts:
lang_filter_status
keep_short                         38997
keep_possible_gaming_or_mixed        752
keep_english                          50
keep_ambiguous                        37
drop_latin_nonenglish_polish          25
drop_latin_nonenglish_german          19
drop_latin_nonenglish_hungarian        6
drop_latin_nonenglish_french           5
drop_latin_nonenglish_turkish          2
drop_latin_nonenglish_spanish          1
drop_latin_nonenglish_bosnian          1
drop_latin_nonenglish_romanian         1
drop_latin_nonenglish_finnish          1
Name: count, dtype: int64

Ambiguous Latin-script language documents kept: 37

Ambiguous documents kept for now:
                                                                                        message  \
1526                                        they have invis tiger i cant spot him on 50 meters   
1942           rinoceronte ty huju zeby ci stara wyruchali w dupsko a potem to wylizała szmata   
2149     

In [12]:
# save to parquet
x_train_lang_clean.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_train.parquet", index=False)

# evaluate impact 
# baseline - before any cleaning
impact = eval_step("non_english_drop", {"WoT": PROJECT_ROOT / "data/processed_data/wot/x_train.parquet"}, impact)
impact

            step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
non_english_drop     WoT 39836    0.5027       0.8699        0.5943           0.4929          0.0097        0.0218              0.0


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,WoT,42960,0.4743,0.8646,0.5476,0.4626,0.0000,0.0000,0.0000
1,non_latin_drop,WoT,39897,0.4930,0.8699,0.5725,0.4929,0.0187,0.0249,0.0303
2,non_english_drop,WoT,39836,0.5027,0.8699,0.5943,0.4929,0.0097,0.0218,0.0000


Increase in metrics are within noise, however they are not negative.

In [13]:
# apply the same Latin-script language filter to the validation set for consistency
x_validation_en_lang = x_validation_en.copy()
x_validation_en_lang["lang_filter_status"] = x_validation_en_lang["message"].apply(latin_language_filter_status)

# inspect groups
print("Language filter status counts:")
print(x_validation_en_lang["lang_filter_status"].value_counts().head(30))


# check ambiguous cases
# these are messages where Lingua did not confidently choose a language
ambiguous_mask = x_validation_en_lang["lang_filter_status"].eq("keep_ambiguous")

print(f"\nAmbiguous Latin-script language documents kept: {ambiguous_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nAmbiguous documents kept for now:\n",
        x_validation_en_lang.loc[
            ambiguous_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among ambiguous rows:")
print(
    x_validation_en_lang.loc[ambiguous_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# check possible gaming / mixed-language false positives
# these are messages that Lingua might classify as non-English, but contain gaming/chat/English cues
mixed_mask = x_validation_en_lang["lang_filter_status"].eq("keep_possible_gaming_or_mixed")

print(f"\nPossible gaming or mixed-language documents kept: {mixed_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nPossible gaming/mixed documents kept for now:\n",
        x_validation_en_lang.loc[
            mixed_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among possible gaming/mixed rows:")
print(
    x_validation_en_lang.loc[mixed_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# drop only confident Latin-script non-English rows
drop_mask = x_validation_en_lang["lang_filter_status"].str.startswith("drop_latin_nonenglish")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nConfident Latin-script non-English documents to drop:\n",
        x_validation_en_lang.loc[
            drop_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print(f"\nHow many confident Latin-script non-English documents: {drop_mask.sum()}")


# label distribution sanity check
dropped_latin_nonenglish = x_validation_en_lang.loc[drop_mask]

print("\nLabel distribution before Latin non-English drop:")
print(x_validation_en["label"].value_counts(normalize=True).sort_index())

print("\nLabel distribution after Latin non-English drop:")
print(
    x_validation_en_lang.loc[~drop_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nLabel distribution among dropped rows:")
print(
    dropped_latin_nonenglish["label"]
    .value_counts(normalize=True)
    .sort_index()
)


# create dataset after Latin non-English drop
# keep ambiguous and mixed/gaming rows
x_validation_lang_clean = (
    x_validation_en_lang
    .loc[~drop_mask]
    .drop(columns=["lang_filter_status"])
    .reset_index(drop=True)
)

print(f"\nRows before Latin non-English drop: {len(x_validation_en_lang)}")
print(f"Dropped {drop_mask.sum()} ({drop_mask.sum() / len(x_validation_en_lang):.1%})")
print(f"Rows after Latin non-English drop: {len(x_validation_lang_clean)}")
# save to parquet
x_validation_lang_clean.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_validation.parquet", index=False)

Language filter status counts:
lang_filter_status
keep_short                         9689
keep_possible_gaming_or_mixed       229
keep_english                         13
drop_latin_nonenglish_german          7
keep_ambiguous                        6
drop_latin_nonenglish_polish          4
drop_latin_nonenglish_hungarian       2
drop_latin_nonenglish_czech           2
drop_latin_nonenglish_french          1
drop_latin_nonenglish_turkish         1
Name: count, dtype: int64

Ambiguous Latin-script language documents kept: 6

Ambiguous documents kept for now:
                                                         message  label  \
2484   thx skorpio but no other help from these line A! campers    2.0   
2592  i did after 1,3 k much too few for a super conqueror, stb    0.0   
4649     pls mommy can I be a motherufcking T34 when i grow up!    1.0   
6160   Enemy SPG <font color='#fe7903'>killerkirby99_2013 (M44)    0.0   
7377        Enemy SPG <font color='#fe7903'>EDELVEIS_UA (FV304)    

#### Duplicates

In [14]:
# look at dataset with duplicates 
x_train_dup = x_train_lang_clean[x_train_lang_clean.duplicated(subset="message", keep=False)].sort_values("message")
# look at some examples
x_train_dup.head(10)

,message,label
28859,!,0.0
4992,!,0.0
11388,!!!!,0.0
2807,!!!!,0.0
32052,#ERROR!,0.0
6079,#ERROR!,0.0
14464,#ERROR!,0.0
14400,#ERROR!,0.0
14397,#ERROR!,0.0
14345,#ERROR!,0.0


In [15]:
# top 100 duplicated messages
x_train_dup["message"].value_counts(ascending=False).head(100)

message
gg                   2556
lol                   354
nice                  312
gj                    257
cap                   162
                     ... 
Im Spotted at F4!      22
Im Spotted at F3!      22
Im Spotted at G5!      21
Im Spotted at J8!      21
??                     21
Name: count, Length: 100, dtype: int64

Most of these duplicates are useful and important data to train or models for detecting toxicity. We need to check for same messages but different labels.

In [16]:
# Look for duplicates with same messages but different labels
conflicts = x_train_lang_clean.groupby("message")["label"].nunique()
conflicts = conflicts[conflicts > 1]

conflict_rows = x_train_lang_clean[x_train_lang_clean["message"].isin(conflicts.index)].sort_values("message")

# Check how large is the proportion of conflicting duplicates
conflict_pct = len(conflict_rows) / len(x_train_lang_clean) * 100
print(f"Proportion of messages with conflicting labels: {conflict_pct:.2f}%")
print(f"Number of messages with conflicting labels: {len(conflict_rows)}")

Proportion of messages with conflicting labels: 14.05%
Number of messages with conflicting labels: 5597


In [17]:
# check conflicting rows 
conflict_rows 

,message,label
15894,#ERROR!,0.0
25435,#ERROR!,0.0
21500,#ERROR!,0.0
11950,#ERROR!,0.0
11985,#ERROR!,0.0
...,...,...
35167,you lost,2.0
21679,you lost,0.0
8596,you lost,1.0
2712,you wanna loose that right?,1.0


These messages can not be used for analysis, since it is a problem of `annotation`. Using them will be equal to guessing. Duplicated rows account for 14% of the data, which is a huge loss but we can not use that.

In [18]:
# drop conflicting messages
x_train_dup = x_train_lang_clean[~x_train_lang_clean["message"].isin(conflicts.index)].reset_index(drop=True)
print(x_train_dup.shape)

(34239, 2)


In [19]:
# save again and evaluate 
x_train_dup.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_train.parquet", index=False)
impact = eval_step("duplicates_removal", {"WoT": PROJECT_ROOT / "data/processed_data/wot/x_train.parquet"}, impact)
impact

              step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
duplicates_removal     WoT 34239    0.4967       0.8667        0.5665           0.5028          -0.006       -0.0278           0.0099


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,WoT,42960,0.4743,0.8646,0.5476,0.4626,0.0000,0.0000,0.0000
1,non_latin_drop,WoT,39897,0.4930,0.8699,0.5725,0.4929,0.0187,0.0249,0.0303
2,non_english_drop,WoT,39836,0.5027,0.8699,0.5943,0.4929,0.0097,0.0218,0.0000
3,duplicates_removal,WoT,34239,0.4967,0.8667,0.5665,0.5028,-0.0060,-0.0278,0.0099


This did not work, our score started decreasing. Instead of dropping duplicated conflicting labels, we assign a majority label to each. 

In [20]:
# majority label for similar words
majority_label = x_train_lang_clean.groupby("message")["label"].agg(lambda x: x.value_counts().index[0])

# cap at 20 occurrences per message — preserves frequency, prevents extreme repeats dominating
MAX_PER_MESSAGE = 20
x_train_deduped = (
    x_train_lang_clean
    .assign(label=lambda df: df["message"].map(majority_label))
    .groupby("message", group_keys=False)
    .apply(lambda g: g.head(MAX_PER_MESSAGE))
    .reset_index(drop=True)
)

# check same ex duplicates
x_train_deduped[x_train_deduped["message"].isin(conflicts.index)][["message", "label"]].head(20)

C:\Users\nyuss\AppData\Local\Temp\ipykernel_31676\984775230.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(MAX_PER_MESSAGE))


,message,label
12,#ERROR!,0.0
13,#ERROR!,0.0
14,#ERROR!,0.0
15,#ERROR!,0.0
16,#ERROR!,0.0
17,#ERROR!,0.0
18,#ERROR!,0.0
19,#ERROR!,0.0
20,#ERROR!,0.0
21,#ERROR!,0.0


In [21]:
# check the change
print(f"Before dedup: {len(x_train_lang_clean)}")
print(f"After dedup: {len(x_train_deduped)}")
dropped = len(x_train_lang_clean) - len(x_train_deduped)
print(f"Dropped: {dropped} ({dropped / len(x_train_lang_clean):.1%})")

Before dedup: 39836
After dedup: 33603
Dropped: 6233 (15.6%)


In [22]:
# save again and evaluate 
x_train_deduped.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_train.parquet", index=False)
impact = eval_step("duplicates_majority", {"WoT": PROJECT_ROOT / "data/processed_data/wot/x_train.parquet"}, impact)
impact

               step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
duplicates_majority     WoT 33603    0.5304       0.8584        0.6166           0.5226          0.0337        0.0501           0.0198


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,WoT,42960,0.4743,0.8646,0.5476,0.4626,0.0000,0.0000,0.0000
1,non_latin_drop,WoT,39897,0.4930,0.8699,0.5725,0.4929,0.0187,0.0249,0.0303
2,non_english_drop,WoT,39836,0.5027,0.8699,0.5943,0.4929,0.0097,0.0218,0.0000
3,duplicates_removal,WoT,34239,0.4967,0.8667,0.5665,0.5028,-0.0060,-0.0278,0.0099
4,duplicates_majority,WoT,33603,0.5304,0.8584,0.6166,0.5226,0.0337,0.0501,0.0198


Cap at 20 gave best f1_macro — higher than both no-dedup and full-dedup. Transform validation data the same way.

In [23]:
# transform validation data as well 
# majority label for similar words 
majority_label = x_validation_lang_clean.groupby("message")["label"].agg(lambda x: x.value_counts().index[0])

# drop duplicates and map majority label 
x_validation_deduped = x_validation_lang_clean.copy()
x_validation_deduped["label"] = x_validation_deduped["message"].map(majority_label)

# save again
x_validation_deduped.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_validation.parquet", index=False)

#### Data Artifacts 


In [24]:
# #ERROR! entries
error_count = x_train_deduped["message"].str.contains(r"#ERROR!", regex=False, na=False).sum()
print(f"#ERROR! messages: {error_count}")

# HTML entities
html_mask = x_train_deduped["message"].str.contains(r"&\w+;", regex=True, na=False)
print(f"Messages with HTML entities: {html_mask.sum()}")
x_train_deduped[html_mask][["message", "label"]].head(10)

#ERROR! messages: 20
Messages with HTML entities: 110


,message,label
56,&gt;.&gt;,0.0
57,&gt;:O,0.0
58,&gt;=o,0.0
59,&lt;-- mostly stock,0.0
60,&lt;3,0.0
61,&lt;3,0.0
62,&lt;3,0.0
63,&lt;3,0.0
64,&lt;3,0.0
65,&lt;3,0.0


In [25]:
# artifact cleaning
# goal: remove generic technical artifacts, not game-specific phrases
# use wot_deduped as input from the previous preprocessing step

x_train_clean = x_train_deduped.copy()

# convert messages to string and fill missing values
x_train_clean["message"] = x_train_clean["message"].fillna("").astype(str)

# decode HTML entities inline
# example: &quot; -> ", &amp; -> &
x_train_clean["message"] = x_train_clean["message"].apply(html_lib.unescape)

# generic artifact patterns
# no game-specific hardcoding here
ERROR_RE = re.compile(r"#ERROR!", flags=re.IGNORECASE)
URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
HTML_TAG_RE = re.compile(r"<[^>]+>", flags=re.IGNORECASE)

# after html.unescape, most entities should disappear
# this catches remaining malformed/unresolved entities
HTML_ENTITY_RE = re.compile(r"&[a-zA-Z]+;|&#\d+;|&#x[0-9a-fA-F]+;")


# classify artifact reason for each row
def artifact_status(text):
    text = str(text)

    if ERROR_RE.search(text):
        return "drop_error"

    if URL_RE.search(text):
        return "drop_url"

    if HTML_TAG_RE.search(text):
        return "drop_html_tag"

    if HTML_ENTITY_RE.search(text):
        return "drop_unresolved_html_entity"

    return "keep"


# apply artifact detection
x_train_clean["artifact_status"] = x_train_clean["message"].apply(artifact_status)

# inspect artifact counts
print("Artifact status counts:")
print(x_train_clean["artifact_status"].value_counts())

# rows to drop
artifact_mask = x_train_clean["artifact_status"].ne("keep")

# inspect dropped artifacts
with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nArtifact rows to drop:\n",
        x_train_clean.loc[
            artifact_mask,
            ["message", "label", "artifact_status"]
        ]
    )

# label distribution sanity check
print("\nLabel distribution before artifact drop:")
print(x_train_deduped["label"].value_counts(normalize=True).sort_index())

print("\nLabel distribution after artifact drop:")
print(
    x_train_clean.loc[~artifact_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nLabel distribution among dropped artifact rows:")
print(
    x_train_clean.loc[artifact_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)

# create artifact-cleaned dataset
x_train_artifact_clean = (
    x_train_clean
    .loc[~artifact_mask]
    .drop(columns=["artifact_status"])
    .reset_index(drop=True)
)

print(f"\nRows before artifact drop: {len(x_train_deduped)}")
print(f"Dropped {artifact_mask.sum()} ({artifact_mask.sum() / len(x_train_deduped):.1%})")
print(f"Rows after artifact drop: {len(x_train_artifact_clean)}")

Artifact status counts:
artifact_status
keep             33563
drop_error          20
drop_html_tag       13
drop_url             7
Name: count, dtype: int64

Artifact rows to drop:
                                                                                                              message  \
12                                                                                                           #ERROR!   
13                                                                                                           #ERROR!   
14                                                                                                           #ERROR!   
15                                                                                                           #ERROR!   
16                                                                                                           #ERROR!   
17                                                                                               

In [26]:
# save again and evaluate 
x_train_artifact_clean.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_train.parquet", index=False)
# baseline - before any cleaning
impact = eval_step("artifact_removal", {"WoT": PROJECT_ROOT / "data/processed_data/wot/x_train.parquet"}, impact)
impact

            step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
artifact_removal     WoT 33563    0.5293       0.8591        0.6138           0.5236         -0.0011       -0.0028            0.001


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,WoT,42960,0.4743,0.8646,0.5476,0.4626,0.0000,0.0000,0.0000
1,non_latin_drop,WoT,39897,0.4930,0.8699,0.5725,0.4929,0.0187,0.0249,0.0303
2,non_english_drop,WoT,39836,0.5027,0.8699,0.5943,0.4929,0.0097,0.0218,0.0000
3,duplicates_removal,WoT,34239,0.4967,0.8667,0.5665,0.5028,-0.0060,-0.0278,0.0099
4,duplicates_majority,WoT,33603,0.5304,0.8584,0.6166,0.5226,0.0337,0.0501,0.0198
5,artifact_removal,WoT,33563,0.5293,0.8591,0.6138,0.5236,-0.0011,-0.0028,0.0010


In [27]:
# apply the same artifact cleaning to validation set for consistency
x_validation_clean = x_validation_lang_clean.copy()

x_validation_clean["message"] = x_validation_clean["message"].fillna("").astype(str)
x_validation_clean["message"] = x_validation_clean["message"].apply(html_lib.unescape)

x_validation_clean["artifact_status"] = x_validation_clean["message"].apply(artifact_status)

x_validation_artifact_clean = (
    x_validation_clean[x_validation_clean["artifact_status"] == "keep"]
    .drop(columns=["artifact_status"])
    .reset_index(drop=True)
)

# save 
x_validation_artifact_clean.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_validation.parquet", index=False)

#### Mislabeled data 

In [28]:
# few top rows of the data
x_train_artifact_clean.head(10)

,message,label
0,!,0.0
1,!,0.0
2,! CAP,0.0
3,!!!!,0.0
4,!!!!,0.0
5,!!!!!!,0.0
6,!Siema kto PL Wn8 (300) IQ 6,0.0
7,"""If you can‘t take a joke, you shouldn‘t be pl...",1.0
8,"""deals too much damage""",0.0
9,"""drives premium tank""",0.0


In [29]:
# class distribution check
print("Label distribution after cleaning:")
print(x_train_artifact_clean["label"].value_counts(normalize=True).sort_index())
print(x_train_artifact_clean["label"].value_counts(normalize=False).sort_index())


Label distribution after cleaning:
label
0.0    0.787891
1.0    0.156661
2.0    0.046807
3.0    0.006316
4.0    0.001609
5.0    0.000715
Name: proportion, dtype: float64
label
0.0    26444
1.0     5258
2.0     1571
3.0      212
4.0       54
5.0       24
Name: count, dtype: int64


We might have mislabeled toxicity data that was classified as non-toxic or opposite. We need to detect and fix that as well. 

In [30]:
from cleanlab.filter import find_label_issues
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict, StratifiedKFold
import numpy as np

# exclude messages under 3 chars — no signal, cleanlab guesses randomly on them
cl_mask = x_train_artifact_clean["message"].str.len() >= 3
cl_df = x_train_artifact_clean[cl_mask].reset_index(drop=True)

X = cl_df["message"].values
y = cl_df["label"].astype(int).values

# stratified cv with shuffle — avoids ordering bias in folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7524)

# lightweight pipeline to generate calibrated out-of-fold probabilities
# class_weight="balanced" — needed with imbalanced data so minority class errors are detected
# LR required (not SVC) — cleanlab needs predict_proba, not decision_function
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95,
                              sublinear_tf=True, norm="l2",
                              analyzer="word", tokenizer=tokenize, token_pattern=None)),
    ("clf", LogisticRegression(max_iter=2000, random_state=7524, class_weight="balanced")),
])

# each sample predicted by model that never saw it during training
oof_probs = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)

# normalized_margin more robust than self_confidence on imbalanced data —
# only flags rows where model is confident another class is correct, not just uncertain
issue_idx = find_label_issues(
    labels=y,
    pred_probs=oof_probs,
    return_indices_ranked_by="normalized_margin",
)

predicted = oof_probs[issue_idx].argmax(axis=1)
given = y[issue_idx]

# only trust 0 - {1,2} boundary crossings —
# class 3/4/5 too rare (< 1%) for reliable OOF predictions, causes kill/die false positives
boundary_mask = (
    ((given == 0) & np.isin(predicted, [1, 2]))
    | (np.isin(given, [1, 2]) & (predicted == 0))
)
issue_idx = issue_idx[boundary_mask]

print(f"Suspected mislabeled: {len(issue_idx)} ({len(issue_idx)/len(y):.1%})")

# inspect top 50 — compare original label vs what model predicted
print(cl_df.iloc[issue_idx[:50]][["message", "label"]].assign(
    predicted=oof_probs[issue_idx[:50]].argmax(axis=1)
))

# map issue positions in cl_df back to x_train_artifact_clean original index
original_idx = x_train_artifact_clean[cl_mask].index[issue_idx]

Suspected mislabeled: 2162 (6.7%)
                           message  label  predicted
18370                     just bot    0.0          1
7730                and an afk bot    0.0          1
28316                      top bot    0.0          1
2005                           Bot    0.0          1
18396        just come help idiots    0.0          1
30942      wtf i win this and loss    0.0          2
28348             top tier top bot    0.0          1
28165                       to bot    0.0          1
9124                 bot peregrine    0.0          1
2004                           Bot    0.0          1
5901               S5 was a bot :/    0.0          1
6502     This Tiger gotta be a bot    0.0          1
13403                  gg bot team    0.0          1
203                          *bots    0.0          1
15808               help your bots    0.0          1
26456        stupid retard campers    0.0          1
12209                    fckg bots    0.0          1
28284       

In [31]:
# drop suspected mislabeled — safer than relabeling with own OOF predictions
# cleanlab paper recommends pruning over relabeling to avoid feedback loop
x_train_artifact_clean1 = x_train_artifact_clean.drop(index=original_idx).reset_index(drop=True)
print(f"Dropped {len(original_idx)} suspected mislabeled rows ({len(original_idx)/len(x_train_artifact_clean):.1%})")

Dropped 2162 suspected mislabeled rows (6.4%)


Cleanlab flagged suspected mislabeled rows using out-of-fold predictions — each sample scored
by a model that never saw it during training. Only 0↔{1,2} boundary crossings are kept:
cases where the model is confident a non-toxic message should be toxic or vice versa.

Flagged rows are **dropped** rather than relabeled. Relabeling with the same model's OOF
predictions creates a feedback loop — the model cleans the data toward its own biases,
then trains on that data. Dropping removes the uncertain examples without introducing
model-generated labels into the training set (Northcutt et al., 2021).

In [32]:
# save again and evaluate 
x_train_artifact_clean1.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_train.parquet", index=False)
impact = eval_step("cleanlab_relabeling", {"WoT": PROJECT_ROOT / "data/processed_data/wot/x_train.parquet"}, impact)
impact

               step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
cleanlab_relabeling     WoT 31401    0.5758       0.9069        0.6496           0.5829          0.0465        0.0358           0.0593


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,WoT,42960,0.4743,0.8646,0.5476,0.4626,0.0000,0.0000,0.0000
1,non_latin_drop,WoT,39897,0.4930,0.8699,0.5725,0.4929,0.0187,0.0249,0.0303
2,non_english_drop,WoT,39836,0.5027,0.8699,0.5943,0.4929,0.0097,0.0218,0.0000
3,duplicates_removal,WoT,34239,0.4967,0.8667,0.5665,0.5028,-0.0060,-0.0278,0.0099
4,duplicates_majority,WoT,33603,0.5304,0.8584,0.6166,0.5226,0.0337,0.0501,0.0198
5,artifact_removal,WoT,33563,0.5293,0.8591,0.6138,0.5236,-0.0011,-0.0028,0.0010
6,cleanlab_relabeling,WoT,31401,0.5758,0.9069,0.6496,0.5829,0.0465,0.0358,0.0593


Gains could be inflated if the model memorizes its own training signal.
Verify with LinearSVC as an independent second model.

In [33]:
from sklearn.svm import LinearSVC
# impact 
impact = eval_step("cleanlab_relabeling_svc", {"WoT": PROJECT_ROOT / "data/processed_data/wot/x_train.parquet"}, impact, 
                                  clf=LinearSVC(max_iter=2000, random_state=7524, 
                                                class_weight = "balanced"))
impact

                   step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
cleanlab_relabeling_svc     WoT 31401    0.6136       0.9183        0.6024           0.6542          0.0378       -0.0472           0.0713


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,WoT,42960,0.4743,0.8646,0.5476,0.4626,0.0000,0.0000,0.0000
1,non_latin_drop,WoT,39897,0.4930,0.8699,0.5725,0.4929,0.0187,0.0249,0.0303
2,non_english_drop,WoT,39836,0.5027,0.8699,0.5943,0.4929,0.0097,0.0218,0.0000
3,duplicates_removal,WoT,34239,0.4967,0.8667,0.5665,0.5028,-0.0060,-0.0278,0.0099
4,duplicates_majority,WoT,33603,0.5304,0.8584,0.6166,0.5226,0.0337,0.0501,0.0198
5,artifact_removal,WoT,33563,0.5293,0.8591,0.6138,0.5236,-0.0011,-0.0028,0.0010
6,cleanlab_relabeling,WoT,31401,0.5758,0.9069,0.6496,0.5829,0.0465,0.0358,0.0593
7,cleanlab_relabeling_svc,WoT,31401,0.6136,0.9183,0.6024,0.6542,0.0378,-0.0472,0.0713


Score gains hold under both LR and LinearSVC. The cleaning is real — label corrections reflect genuine annotation noise, not model self-confirmation.

In [34]:
# fit on full train
pipe.fit(x_train_artifact_clean1["message"], x_train_artifact_clean1["label"])
val_probs = pipe.predict_proba(x_validation_artifact_clean["message"])

# find suspicious val labels — no circularity, model never saw val
val_issues = find_label_issues(
    labels=x_validation_artifact_clean["label"].astype(int).values,
    pred_probs=val_probs,
    return_indices_ranked_by="normalized_margin"
)

val_predicted = val_probs[val_issues].argmax(axis=1)
val_given = x_validation_artifact_clean["label"].astype(int).values[val_issues]

boundary_mask = (
    ((val_given == 0) & np.isin(val_predicted, [1, 2, 3]))
    | (np.isin(val_given, [1, 2, 3]) & (val_predicted == 0))
)
val_issues_filtered = val_issues[boundary_mask]

print(f"Suspected val mislabeled: {len(val_issues_filtered)} ({len(val_issues_filtered)/len(x_validation_artifact_clean):.1%})")

# inspect top 50
print(x_validation_artifact_clean.iloc[val_issues_filtered[:50]][["message", "label"]].assign(
    predicted=val_probs[val_issues_filtered[:50]].argmax(axis=1)
))

Suspected val mislabeled: 652 (6.6%)
                                    message  label  predicted
6345                                 idiots    0.0          1
1263                     damn idiots noone?    0.0          1
8629                                    bot    0.0          1
6319                              its a bot    0.0          1
6191                       top tier top bot    0.0          1
6151                       was that a bot??    0.0          1
9511                                    bot    0.0          1
414                             t1heavy bot    0.0          1
295                                     bot    0.0          1
3560                                    bot    0.0          1
3481                              me bot...    0.0          1
7463                                    bot    0.0          1
2002                         help your bots    0.0          1
2014                                go bots    0.0          1
669                          haft

Cleanlab estimated ~6.7% annotation noise in the validation set using a model trained
on cleaned training data. Clear annotation errors exist: "bot", "idiots", "moron"
labeled as non-toxic (class 0) are genuine annotator mistakes.

**Why val labels are left unchanged:**

Dropping or relabeling val rows based on a train-fitted model is circular — the model
decides what to remove from its own test set, then gets evaluated on the cleaner result.
This inflates metrics without reflecting real improvement.

Val metrics are slightly pessimistic due to annotation noise. This is the honest
evaluation of model performance on real-world noisy data.

#### Final verification

In [35]:
# basic data checks for train and validation sets after cleaning and relabeling
print("Final training set shape:", x_train_artifact_clean1.shape)
print("Final validation set shape:", x_validation_artifact_clean.shape)
print("\nFinal training set label distribution:")
print(x_train_artifact_clean1["label"].value_counts(normalize=True).sort_index())
print("\nFinal validation set label distribution:")
print(x_validation_artifact_clean["label"].value_counts(normalize=True).sort_index())

Final training set shape: (31401, 2)
Final validation set shape: (9899, 2)

Final training set label distribution:
label
0.0    0.794306
1.0    0.154167
2.0    0.042292
3.0    0.006751
4.0    0.001720
5.0    0.000764
Name: proportion, dtype: float64

Final validation set label distribution:
label
0.0    0.813213
1.0    0.136276
2.0    0.043540
3.0    0.004950
4.0    0.001515
5.0    0.000505
Name: proportion, dtype: float64


In [36]:
# save to parquet for easier loading in future steps
# train
x_train_artifact_clean1.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_train.parquet", index=False)
# validation
x_validation_artifact_clean.to_parquet(PROJECT_ROOT / "data/processed_data/wot/x_validation.parquet", index=False)
# save preprocessing ablation results
impact.to_csv(PROJECT_ROOT / "data/results/cleaning_ablation_wot.csv", index=False)
print("Saved WoT cleaning ablation CSV")

Saved WoT cleaning ablation CSV


## Dota

**Dota**
| Class | Label |
|---|---|
| 0 | Other (non-toxic) |
| 1 | Ego |
| 2 | Aggression |
| 3 | Impolite |

#### Inspect each split

In [37]:
# import data
train = pd.read_csv(DATA_DIR_DOTA / "CONDA_train.csv")
val = pd.read_csv(DATA_DIR_DOTA / "CONDA_valid.csv")

# inspect each split
for name, df in [("train", train), ("val", val)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}\n")
    print(f"Columns: {list(df.columns)}\n")
    print(f"First few rows of the data:\n{df.head(3)}\n")

--- train ---
Shape: (26921, 10)

Columns: ['Id', 'matchId', 'conversationId', 'utterance', 'chatTime', 'playerSlot', 'playerId', 'intentClass', 'slotClasses', 'slotTokens']

First few rows of the data:
      Id  matchId  conversationId utterance  chatTime  playerSlot  \
0  11263      697            3193      wow!        76           0   
1  13741      843            3809       WTF      1563           5   
2  22125     1412            6199   wpe wpe      2853           1   

                  playerId intentClass slotClasses          slotTokens  
0  ANTS IN MY EYES JOHNSON           O          O            wow (O),   
1                      M.k           O          T            WTF (T),   
2              Acqua Ragia           O        O O   wpe (O), wpe (O),   

--- val ---
Shape: (8974, 10)

Columns: ['Id', 'matchId', 'conversationId', 'utterance', 'chatTime', 'playerSlot', 'playerId', 'intentClass', 'slotClasses', 'slotTokens']

First few rows of the data:
      Id  matchId  conversa

#### Clean, merge and standardise

In [38]:
# map intentClass to numeric label consistent with WoT schema
label_map = {'O': 0, 'E': 1, 'A': 2, 'I': 3}

# add split column and map labels
train["split"] = "train"
val["split"]   = "val"

# merge train + val (no test split in CONDA)
dota = pd.concat([train, val], ignore_index=True)

# keep only relevant columns and rename to match WoT schema
dota = dota[["Id", "utterance", "intentClass", "split"]].rename(columns={
    "Id": "index",
    "utterance":   "message",
    "intentClass": "label",
})

# map string labels to numeric
dota["label"] = dota["label"].map(label_map)

# drop nulls in message (7 train + 1 val)
dota = dota.dropna(subset=["message"]).reset_index(drop=True)

# shape 
print(f"Merged shape: {dota.shape}\n")
print(f"First few rows of the data:\n{dota.head(3)}\n")
print("\nInformation about the dataset:\n")
dota.info()

Merged shape: (35887, 4)

First few rows of the data:
   index  message  label  split
0  11263     wow!      0  train
1  13741      WTF      0  train
2  22125  wpe wpe      0  train


Information about the dataset:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35887 entries, 0 to 35886
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   index    35887 non-null  int64 
 1   message  35887 non-null  object
 2   label    35887 non-null  int64 
 3   split    35887 non-null  object
dtypes: int64(2), object(2)
memory usage: 1.1+ MB


In [39]:
# X and Y
X = dota["message"]
y = dota["label"]

# split the data into our train and test 
x_train, x_validation, y_train, y_validation = train_test_split(X, y, test_size=0.2, random_state=seed, 
                                             shuffle = True, stratify=y)

# distributions for each split 
print("Distribution of labels in the training set normalized and not-normalized:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))
print("\nDistribution of labels in the validation set normalized and not-normalized:")
print(y_validation.value_counts())
print(y_validation.value_counts(normalize=True))

# save both datasets as one parquet file each, with message and label columns,
# for easier processing in the next steps
x_train = pd.DataFrame({"message": x_train, "label": y_train})
x_validation = pd.DataFrame({"message": x_validation, "label": y_validation})
x_train.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_train.parquet", index=False)
x_validation.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_validation.parquet", index=False)

Distribution of labels in the training set normalized and not-normalized:
label
0    21282
1     3769
2     1839
3     1819
Name: count, dtype: int64
label
0    0.741301
1    0.131283
2    0.064057
3    0.063360
Name: proportion, dtype: float64

Distribution of labels in the validation set normalized and not-normalized:
label
0    5321
1     942
2     460
3     455
Name: count, dtype: int64
label
0    0.741293
1    0.131234
2    0.064085
3    0.063388
Name: proportion, dtype: float64


In [40]:
# do baseline evaluation before any cleaning
impact = None
impact = eval_step("baseline", {"DOTA": PROJECT_ROOT / "data/processed_data/dota/x_train.parquet"}, impact)

    step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
baseline    DOTA 28709    0.7204        0.851        0.7544           0.7013             0.0           0.0              0.0


Only `utterance`, `intentClass`, `Id`, and `split` are kept = the remaining columns (matchId, conversationId, chatTime, playerSlot, playerId, slotClasses, slotTokens) carry no linguistic content relevant to toxicity detection. Labels are mapped to integers matching the WoT schema where 0 = non-toxic. Null utterances are dropped - they carry no text signal.

#### Non-english language 

Since we are going to model classifier on english data, it is important for us to detect non-english cases as many as we can. 

In [41]:
# covers all major non-Latin scripts
NON_LATIN_SCRIPT = re.compile(
    r"[\u0400-\u04FF"   # Cyrillic
    r"\u4E00-\u9FFF"    # CJK unified ideographs
    r"\u3400-\u4DBF"    # CJK extension A
    r"\uF900-\uFAFF"    # CJK compatibility ideographs
    r"\u0600-\u06FF"    # Arabic
    r"\u0590-\u05FF"    # Hebrew
    r"\u3040-\u30FF"    # Japanese (Hiragana + Katakana)
    r"\uAC00-\uD7AF"    # Korean (Hangul syllables)
    r"\u1100-\u11FF"    # Korean (Hangul Jamo)
    r"\u0E00-\u0E7F"    # Thai
    r"\u0900-\u097F"    # Devanagari (Hindi)
    r"\u0980-\u09FF"    # Bengali
    r"\u0370-\u03FF"    # Greek
    r"\u10A0-\u10FF"    # Georgian
    r"\u0530-\u058F"    # Armenian
    r"\u1000-\u109F"    # Myanmar
    r"\u1780-\u17FF]"   # Khmer
)

# drop any message containing even 1 non-Latin script char
print(f"Non-english documents:\n{x_train[x_train['message'].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)}\n")

before = len(x_train)
x_train_en = x_train[~x_train["message"].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)
print(f"Dropped {before - len(x_train_en)} ({(before - len(x_train_en))/before:.1%})")
print("Label distribution before:")
print(x_train["label"].value_counts(normalize=True).sort_index(), "\n")

print("\nLabel distribution after:")
print(x_train_en["label"].value_counts(normalize=True).sort_index(), "\n")
print(f"Rows after the drop: {x_train_en.shape[0]}")
# save new step again
x_train_en.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_train.parquet", index=False)

Non-english documents:
                                     message  label
0                                        Νοο      0
1                                        戒了吧      0
2                                          ツ      0
3                                         ำผ      0
4   dochu....양학....... [SEPA] 일본...재패니즈몽키...      0
5                        s [SEPA] v [SEPA] ツ      0
6                                          ื      0
7                                         ทำ      0
8                                      คนไทย      0
9                                      ใจยเน      0
10                              ไย [SEPA] wp      0
11                                 ¯\_(ツ)_/¯      0
12      sure go mid with bh [SEPA] ¯\_(ツ)_/¯      0
13                                 ¯\_(ツ)_/¯      0
14                           noob [SEPA] กาก      1
15                              เเ [SEPA] gg      0
16                                       กาก      0

Dropped 17 (0.1%)
Label distribution bef

In [42]:
# save and evaluate non-Latin drop
impact = eval_step("non_latin_drop", {"DOTA": PROJECT_ROOT / "data/processed_data/dota/x_train.parquet"}, impact)
impact 

          step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
non_latin_drop    DOTA 28692    0.7225       0.8518        0.7556            0.704          0.0021        0.0012           0.0027


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,DOTA,28709,0.7204,0.8510,0.7544,0.7013,0.0000,0.0000,0.0000
1,non_latin_drop,DOTA,28692,0.7225,0.8518,0.7556,0.7040,0.0021,0.0012,0.0027


In [43]:
# transform that on the validation set as well, to be consistent with the training data.
# drop any message containing even 1 non-Latin script char
print(f"Non-english documents:\n{x_validation[x_validation['message'].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)}\n")

# changes 
before = len(x_validation)
x_validation_en = x_validation[~x_validation["message"].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)
print(f"Dropped {before - len(x_validation_en)} ({(before - len(x_validation_en))/before:.1%})")
print("Label distribution before:")
print(x_validation["label"].value_counts(normalize=True).sort_index(), "\n")

print("\nLabel distribution after:")
print(x_validation_en["label"].value_counts(normalize=True).sort_index(), "\n")
print(f"Rows after the drop: {x_validation_en.shape[0]}")
# save new step again
x_validation_en.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_validation.parquet", index=False)

Non-english documents:
      message  label
0  ညသ တေမိ  ၊      0

Dropped 1 (0.0%)
Label distribution before:
label
0    0.741293
1    0.131234
2    0.064085
3    0.063388
Name: proportion, dtype: float64 


Label distribution after:
label
0    0.741257
1    0.131253
2    0.064094
3    0.063397
Name: proportion, dtype: float64 

Rows after the drop: 7177


Decreases are minor and more likely to be noise. 

In [44]:
from lingua import Language, LanguageDetectorBuilder

# detecting Latin-script but likely non-English messages

# use all Latin-script languages supported by Lingua
# English must stay inside the detector, otherwise English text gets forced into another language
LATIN_LANGUAGES = list(Language.all_with_latin_script())

# detector for Latin-script language detection
# minimum_relative_distance: if top languages are too close, Lingua returns None instead of guessing
detector = (
    LanguageDetectorBuilder
    .from_languages(*LATIN_LANGUAGES)
    .with_minimum_relative_distance(0.25)
    .build()
)

# safety tokens: if these appear, keep the message because it may be gaming/chat English or mixed slang
SAFETY_TOKENS = {
    # gaming / chat
    "gg", "ez", "wp", "gl", "hf", "glhf", "afk", "brb", "lol", "lmao", "lmfao",
    "omg", "wtf", "stfu", "noob", "n00b", "bot", "team", "report", "ban",
    "kick", "mute", "trash", "toxic", "troll", "feed", "feeder", "int", "throw",
    "rank", "elo", "kill", "kys", "die", "fps", "lag", "ping", "server",
    "wot", "dota", "fortnite", "fn",

    # English / toxicity cues
    "you", "your", "you're", "u", "ur", "are", "is", "this", "that", "the",
    "and", "not", "dont", "don't", "fuck", "fucking", "fuk", "fck", "shit",
    "bitch", "ass", "asshole", "idiot", "stupid", "dumb", "moron", "loser",
    "suck", "sucks"
}

TOKEN_RE = re.compile(r"[a-z0-9']+", flags=re.IGNORECASE)


def get_tokens(text):
    return set(TOKEN_RE.findall(str(text).lower()))


def has_safety_token(text):
    return bool(get_tokens(text) & SAFETY_TOKENS)


# function to classify what to do with each Latin-script message
def latin_language_filter_status(text):
    text_str = str(text).strip()
    tokens = get_tokens(text_str)

    # too short - keep
    # language detection is unreliable on short gaming/chat messages
    if len(text_str) < 50 or len(tokens) < 6:
        return "keep_short"

    # possible gaming/chat/English false positive - keep
    if has_safety_token(text_str):
        return "keep_possible_gaming_or_mixed"

    lang = detector.detect_language_of(text_str)

    # ambiguous - keep
    if lang is None:
        return "keep_ambiguous"

    # English - keep
    if lang == Language.ENGLISH:
        return "keep_english"

    # confident Latin-script non-English - drop
    return f"drop_latin_nonenglish_{lang.name.lower()}"


# apply detection
x_train_en_lang = x_train_en.copy()
x_train_en_lang["lang_filter_status"] = x_train_en_lang["message"].apply(latin_language_filter_status)

# inspect groups
print("Language filter status counts:")
print(x_train_en_lang["lang_filter_status"].value_counts().head(30))


# check ambiguous cases
# these are messages where Lingua did not confidently choose a language
ambiguous_mask = x_train_en_lang["lang_filter_status"].eq("keep_ambiguous")

print(f"\nAmbiguous Latin-script language documents kept: {ambiguous_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nAmbiguous documents kept for now:\n",
        x_train_en_lang.loc[
            ambiguous_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among ambiguous rows:")
print(
    x_train_en_lang.loc[ambiguous_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# check possible gaming / mixed-language false positives
# these are messages that Lingua might classify as non-English, but contain gaming/chat/English cues
mixed_mask = x_train_en_lang["lang_filter_status"].eq("keep_possible_gaming_or_mixed")

print(f"\nPossible gaming or mixed-language documents kept: {mixed_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nPossible gaming/mixed documents kept for now:\n",
        x_train_en_lang.loc[
            mixed_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among possible gaming/mixed rows:")
print(
    x_train_en_lang.loc[mixed_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# drop only confident Latin-script non-English rows
drop_mask = x_train_en_lang["lang_filter_status"].str.startswith("drop_latin_nonenglish")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nConfident Latin-script non-English documents to drop:\n",
        x_train_en_lang.loc[
            drop_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print(f"\nHow many confident Latin-script non-English documents: {drop_mask.sum()}")


# label distribution sanity check
dropped_latin_nonenglish = x_train_en_lang.loc[drop_mask]

print("\nLabel distribution before Latin non-English drop:")
print(x_train_en["label"].value_counts(normalize=True).sort_index())

print("\nLabel distribution after Latin non-English drop:")
print(
    x_train_en_lang.loc[~drop_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nLabel distribution among dropped rows:")
print(
    dropped_latin_nonenglish["label"]
    .value_counts(normalize=True)
    .sort_index()
)


# create dataset after Latin non-English drop
# keep ambiguous and mixed/gaming rows
x_train_lang_clean = (
    x_train_en_lang
    .loc[~drop_mask]
    .drop(columns=["lang_filter_status"])
    .reset_index(drop=True)
)

print(f"\nRows before Latin non-English drop: {len(x_train_en)}")
print(f"Dropped {drop_mask.sum()} ({drop_mask.sum() / len(x_train_en):.1%})")
print(f"Rows after Latin non-English drop: {len(x_train_lang_clean)}")

Language filter status counts:
lang_filter_status
keep_short                          26765
keep_possible_gaming_or_mixed        1652
keep_ambiguous                        153
keep_english                          109
drop_latin_nonenglish_tagalog           5
drop_latin_nonenglish_portuguese        4
drop_latin_nonenglish_spanish           3
drop_latin_nonenglish_sotho             1
Name: count, dtype: int64

Ambiguous Latin-script language documents kept: 153

Ambiguous documents kept for now:
                                                                                                                                                                                    message  \
52                                                                                                                                we stack [SEPA] his comp crash [SEPA] pls wait rebooting   
244                                                                                                                    

In [45]:
# save to parquet
x_train_lang_clean.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_train.parquet", index=False)

# evaluate impact 
impact = eval_step("non_english_drop", {"DOTA": PROJECT_ROOT / "data/processed_data/dota/x_train.parquet"}, impact)
impact

            step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
non_english_drop    DOTA 28679     0.718       0.8492        0.7533            0.698         -0.0045       -0.0023           -0.006


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,DOTA,28709,0.7204,0.8510,0.7544,0.7013,0.0000,0.0000,0.0000
1,non_latin_drop,DOTA,28692,0.7225,0.8518,0.7556,0.7040,0.0021,0.0012,0.0027
2,non_english_drop,DOTA,28679,0.7180,0.8492,0.7533,0.6980,-0.0045,-0.0023,-0.0060


In [46]:
# apply the same Latin-script language filter to the validation set for consistency
x_validation_en_lang = x_validation_en.copy()
x_validation_en_lang["lang_filter_status"] = x_validation_en_lang["message"].apply(latin_language_filter_status)

# inspect groups
print("Language filter status counts:")
print(x_validation_en_lang["lang_filter_status"].value_counts().head(30))


# check ambiguous cases
# these are messages where Lingua did not confidently choose a language
ambiguous_mask = x_validation_en_lang["lang_filter_status"].eq("keep_ambiguous")

print(f"\nAmbiguous Latin-script language documents kept: {ambiguous_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nAmbiguous documents kept for now:\n",
        x_validation_en_lang.loc[
            ambiguous_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among ambiguous rows:")
print(
    x_validation_en_lang.loc[ambiguous_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# check possible gaming / mixed-language false positives
# these are messages that Lingua might classify as non-English, but contain gaming/chat/English cues
mixed_mask = x_validation_en_lang["lang_filter_status"].eq("keep_possible_gaming_or_mixed")

print(f"\nPossible gaming or mixed-language documents kept: {mixed_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nPossible gaming/mixed documents kept for now:\n",
        x_validation_en_lang.loc[
            mixed_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among possible gaming/mixed rows:")
print(
    x_validation_en_lang.loc[mixed_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# drop only confident Latin-script non-English rows
drop_mask = x_validation_en_lang["lang_filter_status"].str.startswith("drop_latin_nonenglish")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nConfident Latin-script non-English documents to drop:\n",
        x_validation_en_lang.loc[
            drop_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print(f"\nHow many confident Latin-script non-English documents: {drop_mask.sum()}")


# label distribution sanity check
dropped_latin_nonenglish = x_validation_en_lang.loc[drop_mask]

print("\nLabel distribution before Latin non-English drop:")
print(x_validation_en["label"].value_counts(normalize=True).sort_index())

print("\nLabel distribution after Latin non-English drop:")
print(
    x_validation_en_lang.loc[~drop_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nLabel distribution among dropped rows:")
print(
    dropped_latin_nonenglish["label"]
    .value_counts(normalize=True)
    .sort_index()
)


# create dataset after Latin non-English drop
# keep ambiguous and mixed/gaming rows
x_validation_lang_clean = (
    x_validation_en_lang
    .loc[~drop_mask]
    .drop(columns=["lang_filter_status"])
    .reset_index(drop=True)
)

print(f"\nRows before Latin non-English drop: {len(x_validation_en_lang)}")
print(f"Dropped {drop_mask.sum()} ({drop_mask.sum() / len(x_validation_en_lang):.1%})")
print(f"Rows after Latin non-English drop: {len(x_validation_lang_clean)}")
# save to parquet
x_validation_lang_clean.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_validation.parquet", index=False)

Language filter status counts:
lang_filter_status
keep_short                          6749
keep_possible_gaming_or_mixed        367
keep_ambiguous                        38
keep_english                          20
drop_latin_nonenglish_portuguese       2
drop_latin_nonenglish_tagalog          1
Name: count, dtype: int64

Ambiguous Latin-script language documents kept: 38

Ambiguous documents kept for now:
                                                                                                       message  \
292                                                   whole game [SEPA] 0 [SEPA] léti [SEPA] he thinsk he did   
310                     zzz [SEPA] worst game ever had [SEPA] invoker gets away 1 hp [SEPA] ta get away 1 hit   
845                                                       one [SEPA] lothar [SEPA] my mid [SEPA] arcane boots   
935          below 3k mmr [SEPA] 'omg, he has invis' [SEPA] 'omg, I dunno where to ward, he roams ! [SEPA] XD   
1116                     

#### Duplicates


In [47]:
# look at dataset with duplicates 
x_train_dup = x_train_lang_clean[x_train_lang_clean.duplicated(subset="message", keep=False)].sort_values("message")
# look at some examples
x_train_dup.head(10)

,message,label
20785,!,0
18064,!,0
22928,!,0
25502,!,0
23352,!,0
6331,!,0
12366,!!!!,0
4446,!!!!,0
13881,#NAME?,0
1962,#NAME?,0


In [48]:
# top 100 duplicated messages
x_train_dup["message"].value_counts(ascending=False).head(100)

message
gg        1145
lol        559
?          346
ggwp       298
haha       277
          ... 
why         14
yea         14
OMG         13
..          13
fuck u      13
Name: count, Length: 100, dtype: int64

In [49]:
# Look for duplicates with same messages but different labels
conflicts = x_train_lang_clean.groupby("message")["label"].nunique()
conflicts = conflicts[conflicts > 1]

conflict_rows = x_train_lang_clean[x_train_lang_clean["message"].isin(conflicts.index)].sort_values("message")

# Check how large is the proportion of conflicting duplicates
conflict_pct = len(conflict_rows) / len(x_train_lang_clean) * 100
print(f"Proportion of messages with conflicting labels: {conflict_pct:.2f}%")
print(f"Number of messages with conflicting labels: {len(conflict_rows)}")

Proportion of messages with conflicting labels: 10.99%
Number of messages with conflicting labels: 3153


In [50]:
# check conflicting rows
conflict_rows

,message,label
20812,),0
16958,),0
22886,),0
11905,),0
12214,),0
...,...,...
1558,zz,0
21168,zz,0
7184,zz,2
13082,zz,0


Same as WoT - annotation noise. Dropping would lose training signal. Assign majority label instead.


In [51]:
# majority label for similar words
majority_label = x_train_lang_clean.groupby("message")["label"].agg(lambda x: x.value_counts().index[0])

# cap at 20 occurrences per message — preserves frequency, prevents extreme repeats dominating
MAX_PER_MESSAGE = 20
x_train_deduped = (
    x_train_lang_clean
    .assign(label=lambda df: df["message"].map(majority_label))
    .groupby("message", group_keys=False)
    .apply(lambda g: g.head(MAX_PER_MESSAGE))
    .reset_index(drop=True)
)

# check same ex duplicates
x_train_deduped[x_train_deduped["message"].isin(conflicts.index)][["message", "label"]].head(20)

,message,label
97,),0
98,),0
99,),0
100,),0
101,),0
102,),0
103,),0
104,),0
105,),0
106,),0


In [52]:
# check the change
print(f"Before dedup: {len(x_train_lang_clean)}")
print(f"After dedup: {len(x_train_deduped)}")
dropped = len(x_train_lang_clean) - len(x_train_deduped)
print(f"Dropped: {dropped} ({dropped / len(x_train_lang_clean):.1%})")

Before dedup: 28679
After dedup: 24116
Dropped: 4563 (15.9%)


In [53]:
# save again and evaluate
x_train_deduped.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_train.parquet", index=False)
impact = eval_step("duplicates_majority", {"DOTA": PROJECT_ROOT / "data/processed_data/dota/x_train.parquet"}, impact)
impact

               step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
duplicates_majority    DOTA 24116    0.7143       0.8353        0.7427           0.6986         -0.0037       -0.0106           0.0006


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,DOTA,28709,0.7204,0.8510,0.7544,0.7013,0.0000,0.0000,0.0000
1,non_latin_drop,DOTA,28692,0.7225,0.8518,0.7556,0.7040,0.0021,0.0012,0.0027
2,non_english_drop,DOTA,28679,0.7180,0.8492,0.7533,0.6980,-0.0045,-0.0023,-0.0060
3,duplicates_majority,DOTA,24116,0.7143,0.8353,0.7427,0.6986,-0.0037,-0.0106,0.0006


Cap at 20 applied same as WoT. Transform validation data the same way.

In [54]:
# transform validation data as well 
# majority label for similar words 
majority_label = x_validation_lang_clean.groupby("message")["label"].agg(lambda x: x.value_counts().index[0])

# drop duplicates and map majority label 
x_validation_deduped = x_validation_lang_clean.copy()
x_validation_deduped["label"] = x_validation_deduped["message"].map(majority_label)

# save again
x_validation_deduped.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_validation.parquet", index=False)

#### Data Artifacts


In [55]:
# artifact cleaning for Dota — input: x_train_deduped (after language filter and dedup)
x_train_clean = x_train_deduped.copy()

# convert messages to string and fill missing values
x_train_clean["message"] = x_train_clean["message"].fillna("").astype(str)

# strip [SEPA] conversation separator — data collection artifact, no linguistic value
x_train_clean["message"] = x_train_clean["message"].str.replace(r"\s*\[SEPA\]\s*", " ", regex=True).str.strip()
print(f"[SEPA] remaining after strip: {x_train_clean['message'].str.contains(r'[SEPA]', regex=False).sum()}")

# generic artifact patterns
URL_RE          = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
HTML_ENTITY_RE  = re.compile(r"&[a-zA-Z]+;|&#\d+;|&#x[0-9a-fA-F]+;")
KEYBOARD_MASH_RE = re.compile(r"\S{20,}")


def artifact_status(text):
    text = str(text)
    if URL_RE.search(text):
        return "drop_url"
    if HTML_ENTITY_RE.search(text):
        return "drop_html_entity"
    if KEYBOARD_MASH_RE.search(text):
        return "drop_keyboard_mash"
    return "keep"


# apply artifact detection
x_train_clean["artifact_status"] = x_train_clean["message"].apply(artifact_status)

print("Artifact status counts:")
print(x_train_clean["artifact_status"].value_counts())

# rows to drop
artifact_mask = x_train_clean["artifact_status"].ne("keep")

# inspect dropped artifacts
with pd.option_context("display.max_rows", 50, "display.max_colwidth", 120):
    print("\nArtifact rows to drop:")
    print(x_train_clean.loc[artifact_mask, ["message", "label", "artifact_status"]])

# label distribution sanity check
print("\nLabel distribution before artifact drop:")
print(x_train_deduped["label"].value_counts(normalize=True).sort_index())

print("\nLabel distribution after artifact drop:")
print(x_train_clean.loc[~artifact_mask, "label"].value_counts(normalize=True).sort_index())

# create artifact-cleaned dataset
x_train_artifact_clean = (
    x_train_clean
    .loc[~artifact_mask]
    .drop(columns=["artifact_status"])
    .reset_index(drop=True)
)

print(f"\nRows before artifact drop: {len(x_train_deduped)}")
print(f"Dropped {artifact_mask.sum()} ({artifact_mask.sum() / len(x_train_deduped):.1%})")
print(f"Rows after artifact drop: {len(x_train_artifact_clean)}")

[SEPA] remaining after strip: 0
Artifact status counts:
artifact_status
keep                  24029
drop_keyboard_mash       80
drop_url                  7
Name: count, dtype: int64

Artifact rows to drop:
                                                                                                                       message  \
14                  !!!!!!!!!!!!!!!!!!!!!!!!!!!!! our PA is dead man we won't do stuff and things that would require a dead WK   
39                                                                                                       #concernedstudent1950   
161                                                                                                       ))))))))))))))))))))   
162                                                                   )))))))))))))))))))))))))))))) hello, gays, how are you?   
163                                                                                                  ))))))))))000000000000000   
...           

In [56]:
# save and evaluate
x_train_artifact_clean.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_train.parquet", index=False)
impact = eval_step("artifact_drop", {"DOTA": PROJECT_ROOT / "data/processed_data/dota/x_train.parquet"}, impact)
impact

         step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
artifact_drop    DOTA 24029    0.7301       0.8493        0.7457             0.72          0.0158         0.003           0.0214


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,DOTA,28709,0.7204,0.8510,0.7544,0.7013,0.0000,0.0000,0.0000
1,non_latin_drop,DOTA,28692,0.7225,0.8518,0.7556,0.7040,0.0021,0.0012,0.0027
2,non_english_drop,DOTA,28679,0.7180,0.8492,0.7533,0.6980,-0.0045,-0.0023,-0.0060
3,duplicates_majority,DOTA,24116,0.7143,0.8353,0.7427,0.6986,-0.0037,-0.0106,0.0006
4,artifact_drop,DOTA,24029,0.7301,0.8493,0.7457,0.7200,0.0158,0.0030,0.0214


In [57]:
# apply same artifact cleaning to validation — same functions, no train info used
x_validation_clean = x_validation_deduped.copy()
x_validation_clean["message"] = x_validation_clean["message"].fillna("").astype(str)

# strip [SEPA]
x_validation_clean["message"] = x_validation_clean["message"].str.replace(r"\s*\[SEPA\]\s*", " ", regex=True).str.strip()

x_validation_clean["artifact_status"] = x_validation_clean["message"].apply(artifact_status)
artifact_mask_val = x_validation_clean["artifact_status"].ne("keep")

print(f"Val artifact rows to drop: {artifact_mask_val.sum()}")

x_validation_artifact_clean = (
    x_validation_clean
    .loc[~artifact_mask_val]
    .drop(columns=["artifact_status"])
    .reset_index(drop=True)
)

print(f"Val rows after artifact drop: {len(x_validation_artifact_clean)}")

Val artifact rows to drop: 17
Val rows after artifact drop: 7157


#### Mislabeled data 

In [58]:
# few top rows of the data
x_train_artifact_clean.head(10)

,message,label
0,= =?,0
1,THAT LAST PICK SPEC'' LIKE U KNOW SHIT ROFL,1
2,!,0
3,!,0
4,!,0
5,!,0
6,!,0
7,!,0
8,! min more Rc,2
9,!!,0


In [59]:
# class distribution check
print("Label distribution after cleaning:")
print(x_train_artifact_clean["label"].value_counts(normalize=True).sort_index())
print(x_train_artifact_clean["label"].value_counts(normalize=False).sort_index())


Label distribution after cleaning:
label
0    0.707770
1    0.155396
2    0.074036
3    0.062799
Name: proportion, dtype: float64
label
0    17007
1     3734
2     1779
3     1509
Name: count, dtype: int64


We might have mislabeled toxicity data that was classified as non-toxic or opposite. We need to detect and fix that as well. 

In [60]:
from cleanlab.filter import find_label_issues
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict, StratifiedKFold
import numpy as np

# exclude messages under 3 chars — no signal, cleanlab guesses randomly on them
cl_mask = x_train_artifact_clean["message"].str.len() >= 3
cl_df = x_train_artifact_clean[cl_mask].reset_index(drop=True)

X = cl_df["message"].values
y = cl_df["label"].astype(int).values

# stratified cv with shuffle — avoids ordering bias in folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7524)

# lightweight pipeline to generate calibrated out-of-fold probabilities
# class_weight="balanced" — needed with imbalanced data so minority class errors are detected
# LR required (not SVC) — cleanlab needs predict_proba, not decision_function
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95,
                              sublinear_tf=True, norm="l2",
                              analyzer="word", tokenizer=tokenize, token_pattern=None)),
    ("clf", LogisticRegression(max_iter=2000, random_state=7524, class_weight="balanced")),
])

# each sample predicted by model that never saw it during training
oof_probs = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)

# normalized_margin more robust than self_confidence on imbalanced data —
# only flags rows where model is confident another class is correct, not just uncertain
issue_idx = find_label_issues(
    labels=y,
    pred_probs=oof_probs,
    return_indices_ranked_by="normalized_margin",
)

predicted = oof_probs[issue_idx].argmax(axis=1)
given = y[issue_idx]

# all classes can be added
boundary_mask = (
    ((given == 0) & np.isin(predicted, [1, 2, 3]))
    | (np.isin(given, [1, 2, 3]) & (predicted == 0))
)
issue_idx = issue_idx[boundary_mask]

print(f"Suspected mislabeled: {len(issue_idx)} ({len(issue_idx)/len(y):.1%})")

# inspect top 50 — compare original label vs what model predicted
print(cl_df.iloc[issue_idx[:50]][["message", "label"]].assign(
    predicted=oof_probs[issue_idx[:50]].argmax(axis=1)
))

# map issue positions in cl_df back to x_train_artifact_clean original index
original_idx = x_train_artifact_clean[cl_mask].index[issue_idx]

Suspected mislabeled: 1723 (7.6%)
                                                 message  label  predicted
3626                                               NOOB!      0          1
12491                                            im noob      0          1
1881                                                 EZ)      0          3
8528                                                 ez&      0          3
1896                                                 EZ^      0          3
15674                                            oh shit      0          1
5543                                alive? what the fuck      0          1
4793                             WP GG fuck rustards LOL      0          1
2297                                        GG fuck this      0          1
2077                                               FUCK!      0          1
13132                                just what the fuck?      0          1
5796                            and you can fuck off too      0   

In [61]:
# drop suspected mislabeled — safer than relabeling with own OOF predictions
# cleanlab paper recommends pruning over relabeling to avoid feedback loop
x_train_artifact_clean1 = x_train_artifact_clean.drop(index=original_idx).reset_index(drop=True)
print(f"Dropped {len(original_idx)} suspected mislabeled rows ({len(original_idx)/len(x_train_artifact_clean):.1%})")

Dropped 1723 suspected mislabeled rows (7.2%)


In [62]:
# save again and evaluate 
x_train_artifact_clean1.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_train.parquet", index=False)
impact = eval_step("cleanlab_relabeling", {"DOTA": PROJECT_ROOT / "data/processed_data/dota/x_train.parquet"}, impact)
impact

               step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
cleanlab_relabeling    DOTA 22306    0.8273       0.9161        0.8263            0.832          0.0972        0.0806            0.112


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,DOTA,28709,0.7204,0.8510,0.7544,0.7013,0.0000,0.0000,0.0000
1,non_latin_drop,DOTA,28692,0.7225,0.8518,0.7556,0.7040,0.0021,0.0012,0.0027
2,non_english_drop,DOTA,28679,0.7180,0.8492,0.7533,0.6980,-0.0045,-0.0023,-0.0060
3,duplicates_majority,DOTA,24116,0.7143,0.8353,0.7427,0.6986,-0.0037,-0.0106,0.0006
4,artifact_drop,DOTA,24029,0.7301,0.8493,0.7457,0.7200,0.0158,0.0030,0.0214
5,cleanlab_relabeling,DOTA,22306,0.8273,0.9161,0.8263,0.8320,0.0972,0.0806,0.1120


Gains could be inflated if the model memorizes its own training signal.
Verify with LinearSVC as an independent second model.

In [63]:
from sklearn.svm import LinearSVC
# impact 
impact = eval_step("cleanlab_relabeling_svc", {"DOTA": PROJECT_ROOT / "data/processed_data/dota/x_train.parquet"}, impact, 
                                  clf=LinearSVC(max_iter=2000, random_state=7524, 
                                                class_weight = "balanced"))
impact

                   step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
cleanlab_relabeling_svc    DOTA 22306    0.8477       0.9286         0.824           0.8785          0.0204       -0.0023           0.0465


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline,DOTA,28709,0.7204,0.8510,0.7544,0.7013,0.0000,0.0000,0.0000
1,non_latin_drop,DOTA,28692,0.7225,0.8518,0.7556,0.7040,0.0021,0.0012,0.0027
2,non_english_drop,DOTA,28679,0.7180,0.8492,0.7533,0.6980,-0.0045,-0.0023,-0.0060
3,duplicates_majority,DOTA,24116,0.7143,0.8353,0.7427,0.6986,-0.0037,-0.0106,0.0006
4,artifact_drop,DOTA,24029,0.7301,0.8493,0.7457,0.7200,0.0158,0.0030,0.0214
5,cleanlab_relabeling,DOTA,22306,0.8273,0.9161,0.8263,0.8320,0.0972,0.0806,0.1120
6,cleanlab_relabeling_svc,DOTA,22306,0.8477,0.9286,0.8240,0.8785,0.0204,-0.0023,0.0465


Score gains hold under both LR and LinearSVC. The cleaning is real — label corrections reflect genuine annotation noise, not model self-confirmation.

In [64]:
# fit on full train
pipe.fit(x_train_artifact_clean1["message"], x_train_artifact_clean1["label"])
val_probs = pipe.predict_proba(x_validation_artifact_clean["message"])

# find suspicious val labels — no circularity, model never saw val
val_issues = find_label_issues(
    labels=x_validation_artifact_clean["label"].astype(int).values,
    pred_probs=val_probs,
    return_indices_ranked_by="normalized_margin"
)

val_predicted = val_probs[val_issues].argmax(axis=1)
val_given = x_validation_artifact_clean["label"].astype(int).values[val_issues]

boundary_mask = (
    ((val_given == 0) & np.isin(val_predicted, [1, 2, 3]))
    | (np.isin(val_given, [1, 2, 3]) & (val_predicted == 0))
)
val_issues_filtered = val_issues[boundary_mask]

print(f"Suspected val mislabeled: {len(val_issues_filtered)} ({len(val_issues_filtered)/len(x_validation_artifact_clean):.1%})")

# inspect top 50
print(x_validation_artifact_clean.iloc[val_issues_filtered[:50]][["message", "label"]].assign(
    predicted=val_probs[val_issues_filtered[:50]].argmax(axis=1)
))

Suspected val mislabeled: 529 (7.4%)
                                                message  label  predicted
6904                                      fuck off zeus      0          1
5135                                      WHAT THE FUCK      0          1
3285                                       spectre noob      0          1
3374                       see that shit is just spooky      0          1
1102                                        LOL fucking      0          1
6926                          good game fuck that pudge      0          1
3515                            no wp alche WP? fuck no      0          1
6139                                       so long fuck      0          1
6729                           Very well fucking played      0          1
499        has nuthing to do with how shit tiny is lmao      0          1
2001                                           gg w[ ez      0          3
4202                                    fuck im lagging      0          1
6

## Validation Label Quality — Inspection Only

Cleanlab estimated ~6.7% annotation noise in the validation set using a model trained
on cleaned training data. Clear annotation errors exist: "bot", "idiots", "moron"
labeled as non-toxic (class 0) are genuine annotator mistakes.

**Why val labels are left unchanged:**

Dropping or relabeling val rows based on a train-fitted model is circular — the model
decides what to remove from its own test set, then gets evaluated on the cleaner result.
This inflates metrics without reflecting real improvement.

Val metrics are slightly pessimistic due to annotation noise. This is the honest
evaluation of model performance on real-world noisy data.

#### Final verification

In [65]:
# basic data checks for train and validation sets after cleaning and relabeling
print("Final training set shape:", x_train_artifact_clean1.shape)
print("Final validation set shape:", x_validation_artifact_clean.shape)
print("\nFinal training set label distribution:")
print(x_train_artifact_clean1["label"].value_counts(normalize=True).sort_index())
print("\nFinal validation set label distribution:")
print(x_validation_artifact_clean["label"].value_counts(normalize=True).sort_index())

Final training set shape: (22306, 2)
Final validation set shape: (7157, 2)

Final training set label distribution:
label
0    0.716713
1    0.157715
2    0.066260
3    0.059311
Name: proportion, dtype: float64

Final validation set label distribution:
label
0    0.740673
1    0.131340
2    0.064832
3    0.063155
Name: proportion, dtype: float64


In [66]:
# save to parquet for easier loading in future steps
# train
x_train_artifact_clean1.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_train.parquet", index=False)
# validation
x_validation_artifact_clean.to_parquet(PROJECT_ROOT / "data/processed_data/dota/x_validation.parquet", index=False)

In [67]:
# save preprocessing ablation results
impact.to_csv(PROJECT_ROOT / "data/results/cleaning_ablation_dota.csv", index=False)
print("Saved Dota cleaning ablation CSV")

Saved Dota cleaning ablation CSV
